# V1DD functional metrics — example figures

Population-level visualizations from the 39,407-ROI asset. Each panel
reproduces or extends a result from [de Vries et al. 2019](https://doi.org/10.1038/s41593-019-0550-9);
see [comparability.md](../comparability.md) for where this pipeline diverges.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ASSET = Path("../../results/409828_V1DD_functional_metrics_2026-09-09_01-41-33")
df = pd.read_parquet(ASSET / "stimulus_metrics.parquet")
tc = np.load(ASSET / "tuning_curves.npz", allow_pickle=True)
rf = np.load(ASSET / "receptive_field_maps.npz", allow_pickle=True)
cm = np.load(ASSET / "condition_means.npz", allow_pickle=True)

plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "figure.dpi": 120})


## Population selectivity

Orientation and direction selectivity for windowed drifting gratings.
De Vries report median excitatory DSI ≈ 0.40, OSI ≈ 0.64, gOSI ≈ 0.29.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

for ax, col, label, xlim in zip(axes,
        ["dgw_dsi", "dgw_osi", "dgw_gosi"],
        ["Direction Selectivity", "Orientation Selectivity", "Global OSI"],
        [(0, 1.5), (0, 1.5), (0, 1)]):
    vals = df[col].dropna()
    ax.hist(vals, bins=60, range=xlim, color="#2a7d9a", alpha=0.75, edgecolor="none")
    ax.axvline(vals.median(), color="#b55a3a", lw=1.5, ls="--", label=f"median={vals.median():.3f}")
    ax.set_xlabel(label)
    ax.set_ylabel("ROIs")
    ax.legend(fontsize=8)

fig.suptitle("Population selectivity — windowed drifting gratings (39,407 ROIs)", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


## Direction tuning — example ROIs

Polar plots for three well-tuned neurons at different preferred directions.


In [ ]:
candidates = df[(df.dgw_osi > 0.7) & (df.pika_roi_confidence > 0.9) & (df.snr > 4)].copy()
candidates["pref_bin"] = (candidates.dgw_preferred_dir // 90).astype(int)
examples = candidates.groupby("pref_bin").first().head(3)

dirs_rad = np.deg2rad(tc["directions"])
fig, axes = plt.subplots(1, 3, subplot_kw={"projection": "polar"}, figsize=(10, 3.5))

for ax, (_, row) in zip(axes, examples.iterrows()):
    idx = df.index.get_loc(df[df.roi_key == row.roi_key].index[0])
    sf_idx = int(row.dgw_preferred_sf)
    trials = tc["dgw_trials"][idx, :, sf_idx, :]
    mean_r = np.nanmean(trials, axis=1)
    sem_r = np.nanstd(trials, axis=1) / np.sqrt(np.sum(~np.isnan(trials), axis=1))
    theta = np.append(dirs_rad, dirs_rad[0])
    mu = np.append(mean_r, mean_r[0])

    ax.plot(theta, mu, "o-", lw=1.5, markersize=3, color="#2a7d9a")
    ax.fill_between(theta,
                    np.append(mean_r - sem_r, mean_r[0] - sem_r[0]),
                    np.append(mean_r + sem_r, mean_r[0] + sem_r[0]),
                    alpha=0.2, color="#2a7d9a")
    ax.set_title(f"pref={row.dgw_preferred_dir:.0f}°  OSI={row.dgw_osi:.2f}", fontsize=9, pad=10)
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(1)

fig.suptitle("Direction tuning — three example ROIs", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


## Cortical depth profile

SNR and orientation selectivity as a function of imaging depth.
Depth bins follow the volume/plane lattice: 50 + 96×(vol-1) + 16×plane µm.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)

depth_bins = np.arange(40, 530, 20)
bin_centres = (depth_bins[:-1] + depth_bins[1:]) / 2

for ax, col, label, color in zip(axes,
        ["snr", "dgw_osi"],
        ["SNR", "Orientation Selectivity"],
        ["#2a7d9a", "#b55a3a"]):
    medians = []
    q25, q75 = [], []
    for lo, hi in zip(depth_bins[:-1], depth_bins[1:]):
        vals = df.loc[(df.depth_um >= lo) & (df.depth_um < hi), col].dropna()
        medians.append(vals.median() if len(vals) > 10 else np.nan)
        q25.append(vals.quantile(0.25) if len(vals) > 10 else np.nan)
        q75.append(vals.quantile(0.75) if len(vals) > 10 else np.nan)
    ax.plot(medians, bin_centres, "-", color=color, lw=1.5)
    ax.fill_betweenx(bin_centres, q25, q75, alpha=0.15, color=color)
    ax.set_xlabel(label)
    ax.set_title(label, fontsize=10)

axes[0].set_ylabel("Depth (µm)")
axes[0].invert_yaxis()
fig.suptitle("Metrics across cortical depth — all 39,407 ROIs", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


## Receptive field maps — example grid

ON subfield maps for six ROIs with measured receptive fields.
The black contour marks the 0.25 response-fraction threshold.


In [ ]:
rf_rois = df[df.has_rf_on].sort_values("snr", ascending=False).head(6)

fig, axes = plt.subplots(2, 3, figsize=(10, 5))
for ax, (_, row) in zip(axes.flat, rf_rois.iterrows()):
    idx = df.index.get_loc(row.name)
    on_map = rf["rf_maps"][idx, 0]
    ax.imshow(on_map, aspect="auto", cmap="RdBu_r", vmin=0, vmax=0.6, origin="lower",
              extent=[rf["azimuths"][0], rf["azimuths"][-1],
                      rf["altitudes"][0], rf["altitudes"][-1]])
    ax.contour(rf["azimuths"], rf["altitudes"], on_map,
               levels=[0.25], colors="k", linewidths=0.8)
    ax.set_title(f"col{row.column}/vol{row.volume}/p{row.plane}", fontsize=8)
    ax.tick_params(labelsize=7)

fig.suptitle("ON subfield maps — six highest-SNR ROIs with RF", fontsize=11, y=1.02)
fig.supxlabel("Azimuth (°)", fontsize=9)
fig.supylabel("Altitude (°)", fontsize=9)
plt.tight_layout()
plt.show()


## Retinotopic gradient

RF azimuth plotted against anatomical X position. Only the 7,068 ROIs with
measured ON receptive fields are shown. The gradient reflects the retinotopic
mapping of visual space onto V1.


In [ ]:
has_rf = df[df.has_rf_on].copy()

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(has_rf.roi_x_um_retinotopic, has_rf.azimuth_rf_on,
                c=has_rf.column, cmap="Set2", s=3, alpha=0.4, edgecolors="none")
ax.set_xlabel("Anatomical X (µm, retinotopic frame)")
ax.set_ylabel("RF Azimuth ON (°)")
ax.set_title(f"Retinotopic gradient — {len(has_rf):,} ROIs with ON receptive fields")
handles = [plt.Line2D([0], [0], marker="o", ls="", color=plt.cm.Set2(i / 5), markersize=5)
           for i in range(5)]
ax.legend(handles, [f"Col {i+1}" for i in range(5)], title="Column", fontsize=8, title_fontsize=9)
plt.tight_layout()
plt.show()


## Natural image selectivity

Lifetime sparseness distributions for 118-image and 12-image sets.
**Caution:** sparseness is not comparable across stimulus sets of different size —
the 1 − 1/n normaliser means the same neurons score higher with more images.
See [natural_images.md](../families/natural_images.md).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

for ax, col, label, n_img in zip(axes,
        ["ni_lifetime_sparseness", "ni12_lifetime_sparseness"],
        ["118 images (NI)", "12 images (NI12)"],
        [118, 12]):
    vals = df[col].dropna()
    ax.hist(vals, bins=50, range=(0, 1), color="#2a7d9a", alpha=0.75, edgecolor="none")
    ax.axvline(vals.median(), color="#b55a3a", lw=1.5, ls="--",
               label=f"median={vals.median():.3f}")
    ax.set_xlabel("Lifetime sparseness")
    ax.set_ylabel("ROIs")
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle("Natural image selectivity — sparseness distributions", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


## Running modulation

Correlation of running speed with dF/F traces (`run_corr_dff`). This is finite
for all 39,407 ROIs — unlike `run_mod_dgf`/`run_mod_dgw`, which require the
relevant stimulus to have both running and stationary epochs.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
vals = df.run_corr_dff.dropna()
ax.hist(vals, bins=80, range=(-0.3, 0.5), color="#2a7d9a", alpha=0.75, edgecolor="none")
ax.axvline(vals.median(), color="#b55a3a", lw=1.5, ls="--",
           label=f"median={vals.median():.4f}")
ax.axvline(0, color="grey", lw=0.8, ls=":")
ax.set_xlabel("Pearson r (running speed vs dF/F)")
ax.set_ylabel("ROIs")
ax.set_title(f"Running correlation — all {len(vals):,} ROIs", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
